# 수업 실습: Materials Project · 실험 DB · 상평형도
AI for Materials Science — Hands-on session 1

오늘은 계산 데이터베이스에서 조건에 맞는 재료를 찾아 표로 정리하고, 실험 데이터도 몇 가지 불러오겠습니다.
마지막에는 계산 에너지로 상평형도를 그립니다. 이때 만든 그림과 표가 출석 제출 자료가 됩니다.

## 오늘의 진행

### 0 · 준비
라이브러리를 설치하고, 수업 자료와 MP API 키를 준비합니다.

### A · Materials Project 조회
질문을 검색 조건으로 바꾸는 법을 익히고, 조건에 맞는 재료를 표로 정리합니다.

### B · matminer 실험 DB
출처가 다른 실험 데이터를 같은 방식으로 불러오고, 행 수와 결측값을 점검합니다.

### C · 상평형도
계산 에너지로 Li–Fe–O convex hull을 만들고 그림으로 확인합니다.

### D · 출석 제출
만든 표와 그림을 학번이 붙은 파일로 저장합니다.

---

위에서부터 한 셀씩 차례로 실행해주세요.
앞 셀에서 만든 변수를 뒤 셀에서 그대로 쓰기 때문에, 순서를 건너뛰면 "이름을 찾을 수 없다"는 오류가 납니다.
코드 안의 `##` 줄은 여러분에게 설명하려고 적어 둔 주석이라 실행되지 않습니다.

여기서 다루지 않는 MP API 예제(구조·전자구조·전지·표면·수용액 등)는
같은 폴더의 `2026_2_Hands_on_session1_api_examples.ipynb`에 모아 두었습니다.

## 0. 준비

네 가지를 갖춰 두겠습니다. 라이브러리, 수업 자료, 저장 폴더, 그리고 MP API 키입니다.

### 0-1. 라이브러리 설치
`pymatgen`은 조성·구조·상평형 계산에, `mp_api`는 Materials Project 조회에,
`matminer`는 실험 데이터셋을 불러오는 데 씁니다.
NumPy, pandas, matplotlib은 이들이 함께 끌고 들어옵니다.

In [ ]:
## 맨 앞의 !는 파이썬이 아니라 터미널 명령을 실행하라는 표시입니다.
!pip install -q pymatgen mp_api matminer

### 0-2. 수업 자료 내려받기
오늘 쓰는 데이터는 모두 수업 저장소의 `Data/` 폴더에 들어 있습니다.
계산 자료(`Li-Fe-P-O_entries.json`)와 실험 데이터셋 네 개를 함께 받으므로 수업 중 별도 다운로드가 없습니다.

이미 한 번 받은 뒤에 다시 실행하면 "폴더가 이미 있다"는 메시지가 나오는데, 문제가 아니니 넘어가세요.

In [ ]:
!git clone https://github.com/kwongibaek/MS49900-AI4M.git

### 0-3. 라이브러리 불러오기
설치와 `import`는 다른 단계입니다. 설치는 파일을 놓아두는 일이고, `import`는 지금 이 노트북으로 가져오는 일입니다.

In [ ]:
## 파일 경로, 시간 기록, 키 입력, 함수 서명 확인에 쓰는 파이썬 기본 도구입니다.
import json
import os
import inspect
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass

## np는 수치 계산, pd는 표, plt는 그래프에 씁니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 저장된 계산 자료를 객체로 되살리고, 조성과 상평형을 다루는 pymatgen 도구입니다.
from monty.serialization import loadfn
from pymatgen.core import Composition
from pymatgen.analysis.phase_diagram import PhaseDiagram, PDPlotter

## MP 조회 도구와, 실험 데이터셋을 불러오는 matminer 도구입니다.
from mp_api.client import MPRester
from mp_api.client.routes.materials.summary import SummaryRester
from matminer.datasets import load_dataset, get_available_datasets, get_dataset_description

### 0-4. 파일 경로 정하기
읽을 폴더와 결과를 저장할 폴더를 여기서 한 번만 정합니다.
셀을 실행하면 `Data/` 폴더에 어떤 파일이 들어 있는지 목록이 나옵니다.
목록 대신 오류가 뜨면 0-2의 `git clone` 셀을 실행했는지 확인해주세요.

In [ ]:
## 0-2에서 내려받은 저장소 안의 Data 폴더입니다.
DATA_DIR = Path("MS49900-AI4M/Data")

## 오늘 만드는 표·그림·제출 파일은 모두 여기에 저장됩니다.
OUTPUT = Path("outputs/02_inclass")
OUTPUT.mkdir(parents=True, exist_ok=True)

sorted(path.name for path in DATA_DIR.iterdir())

### 0-5. MP API 키 입력하기
A절과 C절 일부는 Materials Project에 직접 조회합니다.
키는 [MP 계정 페이지](https://next-gen.materialsproject.org/api)에서 무료로 발급받을 수 있습니다.

키는 비밀번호와 같습니다. 코드나 제출 파일에 그대로 적으면 공개되니 조심하세요.
`getpass`는 입력한 글자가 화면에 보이지 않게 받아 줍니다.

In [ ]:
## 환경변수에 키를 넣어 둔 경우에는 그것을 쓰고, 없을 때만 직접 입력받습니다.
API_KEY = os.getenv("MP_API_KEY", "").strip()
if not API_KEY:
    API_KEY = getpass("Materials Project API key: ").strip()

if not API_KEY:
    raise ValueError("MP API 키가 필요합니다. 키를 발급받은 뒤 이 셀을 다시 실행해주세요.")

## 연결이 되는지, 그리고 지금 조회하는 DB 버전이 무엇인지 먼저 확인합니다.
with MPRester(API_KEY) as mpr:
    print("MP DB version:", mpr.db_version)

## A. Materials Project에서 조건에 맞는 재료 찾기

재료 데이터베이스를 쓰는 일은 결국 **"내 질문을 검색 조건으로 바꾸는 일"**입니다.
같은 "LiFePO₄를 보고 싶다"라도, 그 조성의 구조들을 보고 싶은 것인지 그 원소들이 들어간 모든 화합물을 보고 싶은 것인지에
따라 써야 할 조건이 다릅니다.

### 먼저 용어 세 가지를 구분합시다

- **material** — 대표 결정 구조 하나입니다. `mp-19017` 같은 ID가 붙습니다.
- **task** — 개별 DFT 계산 하나입니다. 한 material에 여러 task가 딸려 있습니다.
- **entry** — 조성과 계산 에너지를 함께 담은 객체입니다. 상평형도를 만들 때 씁니다.

같은 조성이라도 원자 배열이 다르면 material이 여러 개입니다.
그리고 summary에 보이는 band gap과 에너지가 항상 **같은 task**에서 나온 값이라고 가정하면 안 됩니다.

### A-1. 질문에 따라 달라지는 검색 조건
아래 다섯 가지가 각각 무엇을 포함하는지 구분해 보세요. 오늘 A절에서 이 중 네 가지를 직접 써 봅니다.

- `formula="LiFePO4"` — 그 조성을 가진 구조 전부. 다형(polymorph)이 여러 개면 여러 개 나옵니다.
- `chemsys="Li-Fe-P-O"` — 정확히 이 네 원소로만 이루어진 화합물.
- `elements=["Li","Fe","P","O"]` — 이 네 원소를 포함. 다른 원소가 더 들어 있어도 걸립니다.
- `get_entries_in_chemsys([...])` — 원소·이원계·삼원계까지 **모든 부분계**. 상평형도를 만들 때 필요합니다.
- `formula="AB"` — 특정 원소를 지정하지 않고 비율만 맞는 조성(anonymous formula). 1:1 화합물을 훑을 때 씁니다.

자세한 문법은 [Querying data](https://docs.materialsproject.org/downloading-data/using-the-api/querying-data)와
[API examples](https://docs.materialsproject.org/downloading-data/using-the-api/examples)에 있습니다.

### A-2. 세 조건의 범위를 눈으로 확인하기
서버에 묻기 전에, 손에 있는 계산 자료로 세 조건이 각각 몇 건을 고르는지 세어 보겠습니다.
여기서 쓰는 파일은 pymatgen이 공개한 Li–Fe–P–O 계산 자료 859건입니다.

숫자가 크게 벌어지는 것을 확인하세요. 조건 하나를 잘못 고르면 보고 싶은 것과 전혀 다른 집합을 보게 됩니다.

In [ ]:
## loadfn은 저장된 JSON을 원래의 entry 객체 목록으로 되살립니다.
reference_entries = loadfn(DATA_DIR / "Li-Fe-P-O_entries.json")

print("계산 자료 entry 수:", len(reference_entries))
print("첫 entry:", reference_entries[0].entry_id, reference_entries[0].composition.reduced_formula)

In [ ]:
## entry 하나에 들어 있는 원소 기호를 중복 없는 집합으로 만들어 주는 함수입니다.
def element_set(entry):
    return {element.symbol for element in entry.composition.elements}

lfp_set = {"Li", "Fe", "P", "O"}

## ==는 원소 집합이 정확히 같은지, <=는 부분집합인지를 검사합니다.
formula_hits = [e for e in reference_entries if e.composition.reduced_formula == "LiFePO4"]
exact_hits = [e for e in reference_entries if element_set(e) == lfp_set]
subsystem_hits = [e for e in reference_entries if element_set(e) <= lfp_set]

pd.DataFrame([
    {"조건": "formula LiFePO4", "이 자료에서 고른 entry 수": len(formula_hits)},
    {"조건": "정확히 Li-Fe-P-O", "이 자료에서 고른 entry 수": len(exact_hits)},
    {"조건": "부분계까지 모두", "이 자료에서 고른 entry 수": len(subsystem_hits)},
])

### A-3. 어떤 조건을 넣을 수 있고, 무엇을 받을 수 있는지 확인하기
검색을 짜기 전에 **넣을 수 있는 조건**과 **받을 수 있는 항목**을 먼저 보는 습관을 들이면 좋습니다.

`inspect.signature`는 함수가 받는 인자 이름을 보여 줍니다. 서버에 연결하지 않고도 확인할 수 있습니다.
`available_fields`는 실제로 받아올 수 있는 항목 이름이라 서버에 물어봅니다.

In [ ]:
## 조회 함수가 받는 조건 이름을 그대로 나열합니다.
print("summary.search에 넣을 수 있는 조건:")
print(list(inspect.signature(SummaryRester.search).parameters))

In [ ]:
## with 블록을 벗어나면 연결이 정리됩니다.
with MPRester(API_KEY) as mpr:
    fields = mpr.materials.summary.available_fields

print("summary에서 받을 수 있는 항목 수:", len(fields))
print(fields[:20], "...")

### A-4. 검색 결과를 표로 바꾸는 함수 만들기
MP는 결과를 문서(document) 객체 목록으로 돌려줍니다. 이걸 매번 손으로 풀어 쓰면 번거로우니,
필요한 항목만 뽑아 DataFrame으로 만들어 주는 함수를 한 번 만들어 두고 A절 내내 재사용하겠습니다.

열 이름에 단위를 같이 적어 두었습니다. band gap은 eV, 에너지는 eV/atom, 밀도는 g/cm³입니다.

In [ ]:
## 조회할 때 요청할 항목입니다. 지정하지 않으면 훨씬 많은 항목이 딸려 옵니다.
SUMMARY_FIELDS = ["material_id", "formula_pretty", "chemsys", "nelements", "nsites",
                  "band_gap", "energy_above_hull", "formation_energy_per_atom",
                  "is_stable", "density", "symmetry"]

## 표에 쓸 열 이름입니다. 단위를 이름에 적어 두면 나중에 헷갈리지 않습니다.
FRAME_COLUMNS = ["material_id", "formula", "chemsys", "nelements", "nsites", "band_gap_eV",
                 "e_hull_eV_atom", "formation_eV_atom", "is_stable", "density_g_cm3",
                 "spacegroup", "spacegroup_number"]

In [ ]:
## 문서 목록을 받아 재료 하나를 한 행으로 갖는 표를 돌려주는 함수입니다.
def summary_to_frame(docs):
    rows = []
    for doc in docs:
        ## symmetry는 값 하나가 아니라 기호와 번호를 품은 객체라, 그 안에서 한 번 더 꺼냅니다.
        sym = doc.symmetry
        rows.append(dict(zip(FRAME_COLUMNS, [
            str(doc.material_id), doc.formula_pretty, doc.chemsys, doc.nelements, doc.nsites,
            doc.band_gap, doc.energy_above_hull, doc.formation_energy_per_atom,
            doc.is_stable, doc.density,
            getattr(sym, "symbol", None), getattr(sym, "number", None),
        ])))
    return pd.DataFrame(rows, columns=FRAME_COLUMNS)

### A-5. LiFePO₄ 조성의 구조들 조회하기
첫 조회입니다. `formula="LiFePO4"`로 이 조성의 구조를 모두 받아 hull 거리 순으로 정렬해 보겠습니다.

`num_chunks=1, chunk_size=200`은 한 번에 최대 200건만 받겠다는 뜻입니다.
그래서 **표의 행 수를 DB 전체 검색 결과 수로 해석하면 안 됩니다.**
행이 200개에 딱 맞게 나왔다면 더 있을 가능성을 의심해야 합니다.

In [ ]:
with MPRester(API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        formula="LiFePO4", fields=SUMMARY_FIELDS,
        all_fields=False, num_chunks=1, chunk_size=200)

lfp_df = summary_to_frame(docs)
lfp_df.to_csv(OUTPUT / "mp_lfp_summary.csv", index=False)

print("받은 행 수:", len(lfp_df))
lfp_df.sort_values("e_hull_eV_atom").head(10)

### A-6. 조건을 겹쳐 후보 좁히기: 1:1 이원 산화물
이번에는 실제 스크리닝처럼 조건을 여러 개 겹쳐 보겠습니다.
찾을 것은 **A와 O가 1:1인 이원 산화물 중, hull에서 0.05 eV/atom 이내이고 band gap이 1–4 eV인 재료**입니다.

- `chemsys="*-O"` — 산소를 포함한 이원계
- `formula="AB"` — 두 원소가 1:1
- `energy_above_hull=(0, 0.05)` — 계산상 안정하거나 매우 가까움
- `band_gap=(1, 4)` — 반도체 영역

`energy_above_hull`이 작다는 것은 이 계산 모델에서 상분해에 대해 안정하다는 뜻일 뿐입니다.
실제로 합성할 수 있는지는 별개의 문제입니다.

In [ ]:
with MPRester(API_KEY) as mpr:
    ao_docs = mpr.materials.summary.search(
        chemsys="*-O", formula="AB", num_elements=2,
        energy_above_hull=(0, 0.05), band_gap=(1, 4),
        fields=SUMMARY_FIELDS, all_fields=False, num_chunks=1, chunk_size=200)

ao_df = summary_to_frame(ao_docs)
ao_df.to_csv(OUTPUT / "mp_AO_candidates.csv", index=False)

print("조건을 통과한 후보:", len(ao_df))
ao_df.sort_values(["e_hull_eV_atom", "band_gap_eV"]).head(15)

### A-7. 서버 조건만으로 부족할 때: R-3m LiXO₂
`formula="ABC2"`는 "세 원소가 1:1:2"라는 뜻일 뿐, **어느 원소가 1이고 어느 원소가 2인지는 정하지 못합니다.**
Li가 2개이고 O가 1개인 조성도 걸립니다.

그래서 두 단계로 나눕니다. 먼저 서버에서 넓게 받고, 받은 결과를 pymatgen `Composition`으로 다시 걸러냅니다.
API로 할 수 있는 것과 손으로 해야 하는 것의 경계를 보여 주는 예입니다.

In [ ]:
## 화학식을 가장 간단한 비로 줄인 뒤 Li가 1개, O가 2개인지 확인합니다.
def is_lixo2(formula):
    comp = Composition(formula).reduced_composition
    ## np.isclose는 소수점 오차를 감안해 "거의 같다"를 판정합니다.
    return (len(comp.elements) == 3 and np.isclose(comp["Li"], 1)
            and np.isclose(comp["O"], 2) and np.isclose(comp.num_atoms, 4))

## 함수가 어떻게 판정하는지 예시로 먼저 확인합니다.
{f: is_lixo2(f) for f in ["LiCoO2", "LiFeO2", "Li2FeO", "LiFePO4"]}

In [ ]:
with MPRester(API_KEY) as mpr:
    ## 공간군 166(R-3m)이고 안정한 3원계 ABC2 후보를 서버에서 먼저 좁힙니다.
    lxo_docs = mpr.materials.summary.search(
        elements=["Li", "O"], num_elements=3, formula="ABC2",
        spacegroup_number=166, is_stable=True, fields=SUMMARY_FIELDS,
        all_fields=False, num_chunks=1, chunk_size=200)

lxo_df = summary_to_frame(lxo_docs)
## map은 formula 열의 값마다 판정 함수를 적용합니다. loc로 True인 행만 남깁니다.
lixo2_df = lxo_df.loc[lxo_df["formula"].map(is_lixo2)].copy()
lixo2_df.to_csv(OUTPUT / "mp_stable_R3m_LiXO2.csv", index=False)

print(f"서버가 돌려준 ABC2 후보 {len(lxo_df)}건 중 실제 LiXO2는 {len(lixo2_df)}건입니다.")
lixo2_df

### A-8. 직접 해보기
A-6의 AO 검색에서 band gap 하한을 `1.0`에서 `2.0`으로 바꿔 실행해 보세요.

- 후보 수가 몇 개로 바뀌었나요?
- 상위 3개의 material ID는 무엇인가요?
- 조건을 더 엄격하게 했을 때 어떤 종류의 재료가 빠졌나요?

조회가 잘 되지 않으면 먼저 B절로 넘어가고, A절 문제는 조교와 함께 해결하면 됩니다.

## B. matminer로 실험 데이터 불러오기

A절에서 본 것은 전부 **계산값**이었습니다. 이번에는 논문에서 수집한 **실험값**을 봅니다.

matminer는 여러 곳에 흩어진 실험 데이터셋을 같은 방식(`load_dataset`)으로 불러오게 해 주는 도구입니다.
matminer 자체가 측정 DB인 것은 아니고, 출처가 제각각인 자료를 하나의 창구로 모아 준다고 생각하면 됩니다.

오늘 볼 네 가지입니다.

- `steel_strength` — 강재 312행. 조성과 항복·인장 강도, 연신율. 강도는 MPa, 조성은 wt%
- `ucsb_thermoelectrics` — 열전 1,093행. **같은 조성이라도 측정 온도가 다르면 다른 행**입니다
- `matbench_expt_gap` — 실험 band gap 4,604행. MP가 배포하지만 DFT 값이 아니라 실험값입니다
- `citrine_thermal_conductivity` — 열전도도 872행. 단위와 측정 조건이 문자열로 섞여 있습니다

이 절의 목표는 물성값 자체보다 **"이 표를 그대로 믿고 계산해도 되는가"를 점검하는 습관**입니다.

### B-1. 어떤 데이터셋이 있는지 둘러보기
`get_available_datasets`로 목록을, `get_dataset_description`으로 설명을 볼 수 있습니다.
관심 주제의 데이터가 이미 있는지 먼저 찾아보는 것이 출발점입니다.

In [ ]:
available = get_available_datasets(print_format=None)
print("matminer가 제공하는 데이터셋 수:", len(available))

## any는 세 검색어 중 하나라도 이름에 들어 있으면 True가 됩니다.
print([name for name in available
       if any(word in name for word in ["expt", "steel", "thermoelectric"])])

In [ ]:
print(get_dataset_description("ucsb_thermoelectrics"))

### B-2. 네 데이터셋 한 번에 불러오기
`load_dataset`에 이름만 주면 표가 됩니다. 출처가 달라도 쓰는 방법은 같습니다.

`data_home`으로 0-2에서 받은 폴더를 지정하고 `download_if_missing=False`를 주었습니다.
수업 중에 인터넷으로 다시 받지 않겠다는 뜻입니다.

In [ ]:
DATASET_NAMES = ["steel_strength", "ucsb_thermoelectrics",
                 "matbench_expt_gap", "citrine_thermal_conductivity"]

## 딕셔너리에 이름을 키로, 표를 값으로 모아 두면 뒤에서 이름으로 꺼내 쓸 수 있습니다.
experimental = {}
for name in DATASET_NAMES:
    experimental[name] = load_dataset(name, data_home=str(DATA_DIR), download_if_missing=False)

## shape는 (행 수, 열 수)입니다.
{name: frame.shape for name, frame in experimental.items()}

### B-3. 네 데이터셋을 한 표로 요약하기
개별 데이터를 들여다보기 전에, 크기·조성 수·결측값·중복을 한눈에 비교해 보겠습니다.
데이터를 처음 받았을 때 가장 먼저 만들면 좋은 표입니다.

In [ ]:
inventory = []
for name, frame in experimental.items():
    ## 데이터셋마다 조성 열 이름이 composition이거나 formula라서 있는 쪽을 고릅니다.
    formula_col = "composition" if "composition" in frame else "formula"
    inventory.append({
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        ## nunique는 중복을 뺀 값의 종류 수입니다.
        "unique_composition_strings": frame[formula_col].nunique(),
        ## isna().sum()을 두 번 하면 표 전체의 빈칸 수가 됩니다.
        "missing_cells": int(frame.isna().sum().sum()),
        ## duplicated()는 모든 열이 똑같은 행을 셉니다.
        "fully_duplicated_rows": int(frame.duplicated().sum()),
    })

inventory_df = pd.DataFrame(inventory)
inventory_df.to_csv(OUTPUT / "experimental_dataset_inventory.csv", index=False)
inventory_df

### B-4. 행 수와 유효 측정값 수는 다릅니다
`len(df)`는 전체 행 수를 셉니다. 하지만 `df.count()`는 **열마다 빈칸을 뺀 개수**를 셉니다.
이 둘이 어디서 갈라지는지 보는 것이 이 절의 핵심입니다.

먼저 강재 데이터를 열어 열 구성부터 확인하겠습니다.

In [ ]:
steel = experimental["steel_strength"]
steel.head()

In [ ]:
## info는 열별 자료형과 값이 들어 있는 행 수를 함께 보여 줍니다.
steel.info()

In [ ]:
## 유효값과 결측값을 열별로 나란히 놓고 비교합니다.
pd.DataFrame({"valid_measurements": steel.count(), "missing": steel.isna().sum()})

`describe()`는 개수·평균·표준편차·분위수를 한 번에 보여 줍니다.
여기서도 `count` 행을 먼저 보세요. 연신율만 개수가 적다면 그 열에 빈칸이 있다는 뜻입니다.

In [ ]:
## 이중 대괄호로 수치 열 세 개만 골라 요약합니다.
steel[["yield strength", "tensile strength", "elongation"]].describe()

### B-5. 같은 조성이 여러 번 나오는 이유
열전 데이터에서는 같은 조성이라도 **측정 온도가 다르면 별개의 기록**입니다.
그래서 "조성이 같으니 중복"이라고 판단해 지워 버리면 데이터를 망칩니다.

아래에서 조성만 같은 행 수와, 모든 열이 같은 완전 중복 행 수를 따로 세어 비교해 보세요.

In [ ]:
te = experimental["ucsb_thermoelectrics"]
te.head()

In [ ]:
## subset을 주면 그 열만 비교하고, 주지 않으면 모든 열이 같아야 중복으로 셉니다.
print("같은 조성이 반복된 행:", int(te.duplicated(subset=["composition"]).sum()))
print("모든 열이 같은 중복 행:", int(te.duplicated().sum()))

## dropna는 지정한 열 중 하나라도 비어 있으면 그 행을 뺍니다.
print("전체 행:", len(te))
print("온도와 zT가 모두 있는 행:", len(te.dropna(subset=["T [K]", "zT"])))

### B-6. 저장된 자료형과 자료의 의미는 다릅니다
pandas의 dtype은 값이 **컴퓨터에 어떻게 저장됐는지**를 알려 줄 뿐,
그 값을 **통계적으로 어떻게 다뤄야 하는지**까지 정해 주지는 않습니다.

- **명목형** — 이름이나 코드이고 순서가 없음 (합성법 이름)
- **순서형** — 순서는 있지만 간격이 일정하다는 보장은 없음 (모스 경도)
- **이산형** — 셀 수 있는 정수량 (구조의 원자 수)
- **연속형** — 구간 안에서 실수값을 가짐 (온도, 측정 물성)

아래 두 셀을 실행한 뒤, `crystallinity` · `synthesis` · `spacegroup` · `T [K]` · `zT`가
각각 어디에 속하는지 짝과 이야기해 보세요.

특히 `spacegroup`은 숫자로 저장되어 있습니다. **그 평균을 계산하면 재료과학적으로 의미가 있을까요?**

In [ ]:
## dtypes는 열별 저장 자료형입니다.
te.dtypes.rename("pandas dtype").to_frame()

In [ ]:
## 저장 형식과 실제 값의 모습을 함께 봅니다. T [K]는 절대온도, zT는 무차원 성능지수입니다.
te[["crystallinity", "synthesis", "spacegroup", "T [K]", "zT"]].head()

### B-7. 측정 조건을 정한 뒤에 비교하기
"zT가 가장 높은 재료"를 그냥 뽑으면 안 됩니다. 온도가 제각각이면 공정한 비교가 아니기 때문입니다.

그래서 먼저 **300–500 K 구간으로 조건을 정하고**, 그 안에서 조성별 최대 zT를 구하겠습니다.
이 값은 "이 구간에서 관측된 최대값"이지 "모든 재료를 같은 온도에서 비교한 결과"가 아닙니다.

In [ ]:
## 필수 열이 빈 행을 먼저 빼고, between으로 양 끝을 포함한 300–500 K를 고릅니다.
te_valid = te.dropna(subset=["T [K]", "zT"]).copy()
te_window = te_valid.loc[te_valid["T [K]"].between(300, 500)]

## groupby는 같은 조성끼리 묶고, agg는 묶음마다 계산할 값을 정합니다.
top_te = (te_window.groupby("composition", as_index=False)
          .agg(n_measurements=("zT", "size"), max_observed_zT=("zT", "max"),
               lowest_T=("T [K]", "min"), highest_T=("T [K]", "max"))
          .sort_values("max_observed_zT", ascending=False))

print(f"300-500 K 측정 기록 {len(te_window)}건, 조성 {len(top_te)}종")
top_te.head(10)

In [ ]:
## fig는 그림 전체, ax는 그래프 영역입니다. 온도 제한 전의 전체 기록을 점으로 찍습니다.
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.scatter(te_valid["T [K]"], te_valid["zT"], s=12, alpha=0.35)
ax.set(xlabel="Measurement temperature (K)", ylabel="Experimental zT",
       title="UCSB thermoelectrics: measured records")

fig.tight_layout()
fig.savefig(OUTPUT / "ucsb_zT_vs_T.png", dpi=160)
plt.show()

### B-8. 실험 band gap 분포 살펴보기
`matbench_expt_gap`은 실험으로 측정한 band gap입니다.
MP가 배포하지만 **DFT 계산값이 아니라 실험값**이라는 점이 중요합니다. A절에서 본 계산 band gap과 섞어 쓰면 안 됩니다.

분포에서 최솟값을 보세요. 0이 많다면 금속으로 측정된 항목이 포함되어 있다는 뜻입니다.

In [ ]:
gap = experimental["matbench_expt_gap"]

## rename으로 요약 결과에 단위가 드러나는 이름을 붙입니다.
gap["gap expt"].describe().rename("Experimental gap (eV)")

### B-9. 무엇을 중복으로 볼지 먼저 정하기
열전도도 데이터에는 단위와 측정 조건이 **문자열로** 섞여 있습니다.
숫자 열만 보고 계산에 넣기 전에 이런 열부터 확인해야 합니다.

In [ ]:
thermal = experimental["citrine_thermal_conductivity"]

## value_counts는 값별 행 수를 셉니다. dropna=False면 빈칸도 함께 셉니다.
thermal["k-units"].value_counts(dropna=False)

In [ ]:
## 두 조건 열을 함께 고른 뒤 같은 조합을 한 번씩만 남겨, 어떤 조건들이 기록되어 있는지 봅니다.
thermal[["k_condition", "k_condition_units"]].drop_duplicates().head(12)

In [ ]:
## 완전 중복과 "formula만 같은 행"을 구분해서 셉니다.
full_duplicate_count = int(thermal.duplicated().sum())
formula_repeat_count = int(thermal.duplicated(subset=["formula"]).sum())

print("모든 열이 같은 중복 행:", full_duplicate_count)
print("formula만 같은 행:", formula_repeat_count)
print("배포본 기준값(18)과 일치:", full_duplicate_count == 18)

완전히 같은 행은 출처를 확인한 뒤 정리할 수 있습니다.
하지만 `formula`만 같은 행은 온도나 측정 조건이 다른 별개의 기록일 수 있으니 한꺼번에 지우면 안 됩니다.

### B-10. 직접 해보기
UCSB 데이터에서 **500–700 K이고 zT ≥ 0.8인 측정 기록**을 뽑아 CSV로 저장하세요.

기록해야 할 숫자는 네 개입니다.

1. 전체 행 수
2. 필수 열의 결측값을 제거한 뒤의 행 수
3. 두 조건을 모두 통과한 행 수
4. 통과한 기록에 들어 있는 서로 다른 조성 문자열 수

그리고 **같은 조성의 여러 행을 남겨 둔 이유**를 한 문장으로 적으세요.

아래 셀에 재료를 준비해 두었습니다. 빈칸을 채우는 문제가 아니라 직접 이어 쓰는 과제입니다.
막히면 B-5와 B-7에서 쓴 `dropna(subset=...)`와 boolean mask를 다시 보세요.

In [ ]:
## 원본 te는 그대로 두고 복사본으로 작업합니다.
task_input = te.copy()
required_cols = ["composition", "T [K]", "zT"]
task_temperature_range = (500, 700)
task_min_zT = 0.8
task_output_csv = OUTPUT / "task_ucsb_500_700K.csv"

## 순서: len() -> dropna(subset=...) -> 두 조건 mask -> nunique() -> to_csv()
## 아래부터 직접 작성하세요.
task_input[required_cols].head()

## C. 계산 에너지로 상평형도 만들기

여기가 오늘의 핵심입니다. 조성별 계산 에너지에서 **어떤 상이 안정한지**를 이끌어내는 과정을 직접 만들어 봅니다.

만들 것은 Li–Fe–O 삼원계 상평형도입니다. 원소를 바꾸면 같은 코드로 Li–Co–O도 그릴 수 있습니다.

### 왜 `chemsys="Li-Fe-O"` 조회만으로는 안 되는가
정확히 세 원소인 화합물만 가져오면 Li, Fe, O₂ 같은 **원소 기준상**과 Li₂O, Fe₂O₃ 같은 **이원계 경쟁상**이 빠집니다.
비교 대상이 없으면 convex hull을 만들 수 없습니다. `is_stable=True`로 걸러도 마찬가지입니다.
불안정상이 없으면 "얼마나 불안정한가"를 잴 수 없기 때문입니다.

그래서 상평형도에는 **원소·이원계·삼원계를 모두 포함한 부분계 전체**가 필요합니다.
A-1에서 본 `get_entries_in_chemsys`가 그 역할을 합니다.

### C-1. 상평형도에 쓸 entry 고르기
오늘은 A-2에서 읽어 둔 계산 자료에서 Li·Fe·O 부분계만 골라 씁니다.
버전이 고정된 자료라 수업 중 모두가 같은 그림을 보게 됩니다.

출력에서 **원소 reference가 세 개 모두 있는지** 확인하세요. 하나라도 빠지면 hull을 만들 수 없습니다.

In [ ]:
PD_ELEMENTS = ["Li", "Fe", "O"]

## A-2에서 만든 element_set 함수를 다시 씁니다. 원소 집합이 Li-Fe-O 안에 들어오는 entry만 남깁니다.
pd_entries = [e for e in reference_entries if element_set(e) <= set(PD_ELEMENTS)]
phase_source = "pymatgen historical fixture 0428f232a569 | stored GGA/GGA+U corrections"

print("이 계의 entry 수:", len(pd_entries))
print("원소 reference:", sorted({e.composition.reduced_formula
                                for e in pd_entries if e.composition.is_element}))

### C-2. convex hull 계산하고 세 에너지 구분하기
`PhaseDiagram`에 entry 목록을 넣으면 convex hull과 안정상 집합을 계산합니다.

표에 에너지 열이 세 개 나옵니다. **각각 무엇인지 구분해서 보세요.**

- `corrected_energy_eV_atom` — 그 계산 자체의 원자당 에너지. 물질끼리 직접 비교하면 안 됩니다
- `formation_energy_eV_atom` — 원소 기준상 대비 형성 에너지. 음수일수록 원소들보다 안정
- `energy_above_hull_eV_atom` — hull까지의 거리. **0이면 이 계에서 안정**, 클수록 분해되기 쉬움

여기서 형성 에너지는 0 K 전자구조 계산의 값입니다.
고체의 `pV` 항이 작다고 보면 0 K 형성 엔탈피의 근사로 쓸 수 있지만, 유한 온도의 Gibbs 자유에너지와는 다릅니다.
온도 의존 자유에너지는 03 노트북에서 다룹니다.

In [ ]:
## 입력 entry의 조성과 보정 에너지로 이 계의 convex hull을 계산합니다.
phase_diagram = PhaseDiagram(pd_entries)

phase_rows = []
for entry in pd_entries:
    phase_rows.append({
        "entry_id": str(entry.entry_id),
        "formula": entry.composition.reduced_formula,
        "n_atoms_in_entry": entry.composition.num_atoms,
        "corrected_energy_eV_atom": entry.energy_per_atom,
        "formation_energy_eV_atom": phase_diagram.get_form_energy_per_atom(entry),
        "energy_above_hull_eV_atom": phase_diagram.get_e_above_hull(entry),
        ## in은 이 entry가 안정상 집합에 들어 있는지 검사합니다.
        "stable_in_this_PD": entry in phase_diagram.stable_entries,
        "run_type": entry.parameters.get("run_type"),
    })

phase_df = pd.DataFrame(phase_rows).sort_values(["energy_above_hull_eV_atom", "formula"])

print("안정 entry 수:", int(phase_df["stable_in_this_PD"].sum()), "/", len(phase_df))
phase_df.head(20)

### C-3. 상평형도 그리기
`PDPlotter`가 hull을 삼각형 위에 그려 줍니다. `show_unstable=False`로 안정상만 남겼습니다.

그림 읽는 법입니다.

- **꼭짓점** — 순수 원소
- **변** — 이원계 화합물
- **내부** — 삼원계 화합물
- **선으로 연결된 상들** — 그 사이 조성에서 함께 존재할 수 있는 조합

이 그림은 고정된 계산 모델의 0 K 에너지가 기준입니다. 온도, 압력, 반응 속도는 들어 있지 않습니다.

In [ ]:
plotter = PDPlotter(phase_diagram, backend="matplotlib", show_unstable=False,
                    linewidth=1.5, markersize=7)
## 기본 이름표를 끄고, 아래에서 위치를 직접 지정해 겹치지 않게 붙입니다.
ax = plotter.get_plot(label_stable=False, label_unstable=False)
ax.set_aspect("equal", adjustable="box")

phase_fig = ax.figure
phase_fig.set_size_inches(8, 7)

In [ ]:
## 가까운 상끼리 이름표가 겹치지 않도록 Li-Fe-O 예제에 맞춰 이동량을 정해 둡니다.
## hull 좌표와 계산값은 건드리지 않고 이름표 위치만 바꿉니다.
label_offsets = {"Li": (-12, -13), "Fe": (14, -13), "O2": (0, 20),
                 "Li2O": (-35, -8), "Li2O2": (-45, 10), "Li2FeO3": (-30, 27),
                 "LiFeO2": (0, 30), "Li2FeO2": (8, -25), "Li5FeO4": (-55, 12),
                 "FeO": (28, -8), "Fe3O4": (40, 5), "Fe2O3": (32, 25)}

## pd_plot_data[1]은 안정상의 좌표와 entry를 짝지어 담고 있습니다.
for xy, entry in plotter.pd_plot_data[1].items():
    formula = entry.composition.reduced_formula
    ## xy는 상의 좌표, xytext는 화면상 이동량(포인트 단위)입니다.
    ax.annotate(formula, xy=xy, xytext=label_offsets.get(formula, (10, 10)),
                textcoords="offset points", ha="center", va="center", fontsize=10,
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.8, pad=1),
                arrowprops=dict(arrowstyle="-", color="0.5", lw=0.6), annotation_clip=False)

ax.set_title("Li-Fe-O | corrected GGA/GGA+U | historical reference")
## bbox_inches="tight"는 바깥으로 나간 이름표까지 포함해 저장합니다.
phase_fig.savefig(OUTPUT / "phase_diagram.png", dpi=180, bbox_inches="tight")
plt.show()

### C-4. 같은 화학식, 다른 구조: LiFeO₂ 다형
그림에는 안정상만 나왔지만, 실제 자료에는 같은 LiFeO₂ 조성의 구조가 여러 개 들어 있습니다.

이들의 hull 거리를 비교하면 **어떤 구조가 실제로 안정한 것인지**가 드러납니다.
그리고 불안정한 구조가 어떤 상으로 분해되는지도 함께 볼 수 있습니다.

In [ ]:
same_formula = [e for e in pd_entries if e.composition.reduced_formula == "LiFeO2"]

lifeo2_rows, lifeo2_product_ids = [], set()
for entry in same_formula:
    ## 이 함수는 분해 생성물과 hull 거리를 한 번에 돌려줍니다.
    decomposition, e_hull = phase_diagram.get_decomp_and_e_above_hull(entry)
    products = [f"{e.entry_id}:{e.composition.reduced_formula}" for e in decomposition]
    lifeo2_product_ids.update(str(e.entry_id) for e in decomposition)
    lifeo2_rows.append({"entry_id": str(entry.entry_id),
                        "energy_above_convex_hull_eV_atom": float(e_hull),
                        "decomposition_products": " + ".join(products)})

lifeo2_polymorphs = (pd.DataFrame(lifeo2_rows)
                     .sort_values("energy_above_convex_hull_eV_atom")
                     .reset_index(drop=True))

print("LiFeO2 다형 수:", len(lifeo2_polymorphs))
print("분해 생성물 ID:", lifeo2_product_ids)
lifeo2_polymorphs

표에서 확인할 것이 두 가지입니다.

첫째, 맨 위 행의 hull 거리는 0입니다. 이 구조가 안정상이고, 그림에 이름이 올라간 LiFeO₂입니다.
맨 아래 행은 0.34 eV/atom 남짓 떨어져 있습니다. 같은 화학식인데도 이만큼 차이가 납니다.

둘째, 불안정한 다형들이 **모두 같은 하나의 상(`mp-851027`)으로 분해**됩니다.
조성이 같으니 분해 상대도 같은 것입니다. 다음 셀에서는 조성이 다를 때 어떻게 달라지는지 봅니다.

### C-5. 조성이 다르면 생성물도 여러 개: LiFe₂O₄
`mp-25516`(LiFe₂O₄)은 hull에서 조금 떨어져 있고, **세 개의 상으로 나뉘어 분해**됩니다.

`get_decomp_and_e_above_hull`이 돌려주는 가중치는 **원자 분율**입니다.
반응식 계수와는 다르니, 아래에서 화학식당 계수로 환산해 보겠습니다.

In [ ]:
## 목록에서 조건에 맞는 첫 entry를 꺼냅니다.
target = next(e for e in pd_entries if str(e.entry_id) == "mp-25516")
decomposition, target_e_hull = phase_diagram.get_decomp_and_e_above_hull(target)

## 약분한 화학식 기준의 원자 수입니다. LiFe2O4는 7개입니다.
target_atoms = Composition(target.composition.reduced_formula).num_atoms

decomposition_rows = []
for product, weight in decomposition.items():
    product_atoms = Composition(product.composition.reduced_formula).num_atoms
    decomposition_rows.append({
        "phase_entry_id": str(product.entry_id),
        "phase_formula": product.composition.reduced_formula,
        "atomic_fraction_weight": float(weight),
        ## 원자 분율 x 대상 원자 수 / 생성물 원자 수 = 화학식당 계수
        "coefficient_per_target_formula": float(weight) * target_atoms / product_atoms,
    })

print(f"{target.entry_id} ({target.composition.reduced_formula}) "
      f"| hull 위 {float(target_e_hull):.6f} eV/atom | 원자 수 {target_atoms:.0f}")
pd.DataFrame(decomposition_rows).sort_values("phase_entry_id")

계수 열을 읽으면 균형 잡힌 분해 반응식이 나옵니다.

$$\mathrm{LiFe_2O_4 \rightarrow LiFeO_2 + \tfrac{1}{2}\,Fe_2O_3 + \tfrac{1}{4}\,O_2}$$

원자 분율 `4/7`, `5/14`, `1/14`에 각각 `7 / 생성물 원자 수`(4, 5, 2개)를 곱하면 1, 1/2, 1/4가 됩니다.

반응식 객체로 균형을 다시 확인하고 eV/reaction 단위로 바꾸는 방법은 03 노트북에서 다룹니다.

## D. 출석 제출

지금까지 만든 표와 그림을 학번이 붙은 파일로 저장합니다. 제출물은 세 개입니다.

1. `attendance_<학번>_phase_diagram.png` — C-3에서 그린 상평형도
2. `attendance_<학번>_phases.csv` — C-2에서 만든 상평형 표
3. `attendance_<학번>_reflection.json` — 아래에서 직접 작성할 설명

### 작성할 내용
- 안정상 3개를 골라 `selected_phase_ids`에 적으세요.
- LiFeO₂ 다형 중 하나를 골라 entry ID와 hull 거리를 적으세요 (C-4 표 참고).
- 상평형도에 **부분계까지 넣어야 하는 이유**를 본인 문장으로 적으세요.
- **0 K convex hull이 설명하지 못하는 현상** 한 가지를 적으세요.

아래 점검 코드는 파일이 만들어졌는지만 확인합니다. 설명 내용을 대신 채워 주지는 않습니다.

In [ ]:
## 본인 학번으로 바꿔주세요.
STUDENT_ID = "demo"

REFLECTION = {
    "why_subsystems": "여기에 자신의 설명을 작성하세요.",
    "one_limitation_of_0K_hull": "여기에 자신의 설명을 작성하세요.",
    "selected_phase_ids": [],
    "lifeo2_entry_id": "직접 선택한 entry ID",
    "lifeo2_e_hull_eV_atom": None,
}

In [ ]:
## 파일명에 쓸 수 없는 문자를 걸러냅니다. 남는 것이 없으면 demo를 씁니다.
safe_student_id = "".join(c for c in STUDENT_ID if c.isalnum() or c in "_-") or "demo"

attendance_csv = OUTPUT / f"attendance_{safe_student_id}_phases.csv"
attendance_png = OUTPUT / f"attendance_{safe_student_id}_phase_diagram.png"
attendance_json = OUTPUT / f"attendance_{safe_student_id}_reflection.json"

phase_df.to_csv(attendance_csv, index=False)
phase_fig.savefig(attendance_png, dpi=180, bbox_inches="tight")

## 나중에 결과를 되짚을 수 있도록 계산 조건과 시각도 함께 기록합니다.
attendance = {
    "student_id": STUDENT_ID,
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "chemical_system": PD_ELEMENTS,
    "source": phase_source,
    "n_entries": len(pd_entries),
    "n_stable": len(phase_diagram.stable_entries),
    "energy_units": "eV/atom",
    "reflection": REFLECTION,
}
## ensure_ascii=False로 두면 한글이 그대로 저장됩니다.
attendance_json.write_text(json.dumps(attendance, ensure_ascii=False, indent=2), encoding="utf-8")

for file in [attendance_png, attendance_csv, attendance_json]:
    print(f"{'생성됨' if file.exists() and file.stat().st_size > 0 else '없음  '}  {file}")

In [ ]:
## 계산이 제대로 됐는지 확인하는 두 가지입니다.
## hull 거리는 음수가 될 수 없고, 원소 기준상은 원소 수만큼 있어야 합니다.
print("hull 거리에 음수 없음:", bool(phase_df["energy_above_hull_eV_atom"].ge(-1e-8).all()))
print("원소 기준상 수:", len(phase_diagram.el_refs), "/", len(PD_ELEMENTS))
print("학번 입력됨:", STUDENT_ID != "demo")
print("설명 작성됨:", not REFLECTION["why_subsystems"].startswith("여기에"))

### 수업 후 과제
관심 있는 재료군에서 조건에 맞는 후보를 좁히고, 선택 근거와 분석의 한계를 함께 보고하세요.

- A-6의 AO 예제를 다른 조성군이나 다른 band gap 구간으로 바꾸고, 안정성을 포함해 조건을 두 개 이상 적용하세요.
- API 접속이 어려우면 `steel_strength` 또는 UCSB 데이터에서 물성과 측정 조건을 두 개 이상 적용하세요.
- 제출물은 실행된 노트북, 후보 CSV, 선택 근거 5–8문장입니다.
  근거에는 원본 행 수, 결측값 처리 뒤의 행 수, 조건 통과 행 수, 단위를 반드시 포함하세요.
- 평가 기준(안): 재현 가능한 코드 40%, 조건과 단위의 타당성 30%, 결과 해석과 한계 30%.

과제를 마친 뒤에는 `2026_2_Hands_on_session1_api_examples.ipynb`의 API 예제나
03 Advanced 노트북을 선택해서 진행할 수 있습니다. Advanced는 성적에 반영하지 않습니다.